In [1]:
"""
cProfile Demo — Applied Parallel Programming (CSC14116)
========================================================
Task: Matrix Multiplication  C = A @ B

How to run:
  - In Colab / Jupyter: change N below, then run the cell.
  - In terminal:        python profiling_demo.py
"""

import cProfile
import pstats
import io
import os
import time
import numpy as np

# ── Configuration ─────────────────────────────────────────────────────────────
N = 128   # matrix size N×N  — try 128 (fast) → 256 → 512 (slow)
# ─────────────────────────────────────────────────────────────────────────────


# ── Implementations ───────────────────────────────────────────────────────────

def matmul_naive(A, B):
    """
    Naive triple-nested loop.
    C[i, j] = sum over k of A[i, k] * B[k, j]

    This is the textbook definition — correct, but very slow in Python.
    The profiler will point straight here.
    """
    n = A.shape[0]
    C = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(n):
            for k in range(n):
                C[i, j] += A[i, k] * B[k, j]
    return C


def matmul_numpy(A, B):
    """
    NumPy matrix multiply — same result, uses optimised BLAS under the hood.
    This is what your GPU kernel should aim to match or beat.
    """
    return A @ B


def verify(C1, C2, tol=1e-3):
    """Check that two results are numerically close."""
    return np.allclose(C1, C2, atol=tol)


def benchmark(fn, A, B, n_runs=3):
    """Run fn(A, B) n_runs times, return median wall-clock time in seconds."""
    times = []
    result = None
    for _ in range(n_runs):
        t0     = time.perf_counter()
        result = fn(A, B)
        times.append(time.perf_counter() - t0)
    return float(np.median(times)), result

In [2]:
# ── Technique 1: cProfile ─────────────────────────────────────────────────────
def demo_cprofile(A, B):
    print("\n" + "=" * 65)
    print("TECHNIQUE 1 — cProfile")
    print("=" * 65)
    print("Which functions cost the most cumulative time?\n")

    profiler = cProfile.Profile()
    profiler.runcall(matmul_naive, A, B)

    stream = io.StringIO()
    stats  = pstats.Stats(profiler, stream=stream)
    stats.strip_dirs()
    stats.sort_stats("cumulative")
    stats.print_stats(10)

    output = stream.getvalue()
    print(output)

    # Save for the proposal submission
    os.makedirs("benchmarks", exist_ok=True)
    with open("benchmarks/profile_output.txt", "w") as f:
        f.write(output)
    print("  Saved → benchmarks/profile_output.txt")


In [3]:
# ── Technique 2: per-function timer ───────────────────────────────────────────
def demo_manual_timer(A, B):
    print("\n" + "=" * 65)
    print("TECHNIQUE 2 — Compare naive vs NumPy")
    print("=" * 65)
    print("How much faster is the optimised version?\n")

    t_naive, C_naive   = benchmark(matmul_naive, A, B, n_runs=3)
    t_numpy, C_numpy   = benchmark(matmul_numpy, A, B, n_runs=3)
    correct            = verify(C_naive, C_numpy)
    speedup            = t_naive / t_numpy if t_numpy > 0 else float("inf")

    print(f"  {'Implementation':<20} {'Time (s)':>10}  {'Speedup':>10}")
    print("  " + "-" * 44)
    print(f"  {'matmul_naive':<20} {t_naive:>10.4f}  {'1.0x (baseline)':>10}")
    print(f"  {'matmul_numpy':<20} {t_numpy:>10.4f}  {speedup:>9.1f}x")
    print("  " + "-" * 44)
    print(f"\n  Correctness check: {'PASS' if correct else 'FAIL'}")
    print(f"\n  → NumPy is {speedup:.0f}x faster than the naive loop.")
    print("    A GPU kernel should aim for a similar (or larger) gap.")
    print("    The profiler in Technique 1 tells you exactly why.")

In [4]:
# ── Technique 3: benchmark() for the proposal ─────────────────────────────────
def demo_benchmark(A, B):
    print("\n" + "=" * 65)
    print("TECHNIQUE 3 — benchmark()")
    print("=" * 65 + "\n")

    elapsed, _ = benchmark(matmul_naive, A, B, n_runs=3)
    n          = A.shape[0]
    flops      = 2 * n ** 3          # multiply-add per output element
    gflops     = flops / elapsed / 1e9

    print(f"  Input:      A = {A.shape}, B = {B.shape}  ({A.dtype})")
    print(f"  Time:       {elapsed:.4f} s  (median of 3 runs)")
    print(f"  Throughput: {gflops:.4f} GFLOP/s")
    print()
    print("  Your GPU speedup target is relative to these numbers.")
    print("  A well-optimised GPU kernel should reach 100+ GFLOP/s.")

In [5]:
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

demo_cprofile(A, B)
demo_manual_timer(A, B)
demo_benchmark(A, B)


TECHNIQUE 1 — cProfile
Which functions cost the most cumulative time?

         545 function calls (539 primitive calls) in 1.800 seconds

   Ordered by: cumulative time
   List reduced from 145 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        3    0.000    0.000    1.800    0.600 base_events.py:1922(_run_once)
        3    0.000    0.000    1.784    0.595 selectors.py:451(select)
        3    0.219    0.073    1.784    0.595 {method 'poll' of 'select.epoll' objects}
        1    1.005    1.005    1.005    1.005 {built-in method time.sleep}
        1    0.560    0.560    0.560    0.560 4292165898.py:25(matmul_naive)
        3    0.000    0.000    0.017    0.006 events.py:86(_run)
        3    0.000    0.000    0.017    0.006 {method 'run' of '_contextvars.Context' objects}
        2    0.000    0.000    0.016    0.008 asyncio.py:206(_handle_events)
        2    0.000    0.000    0.016    0.008 zmqstream.py:574(_handle_event